# SSURGO Soil Data Download — Iowa

Downloads Iowa soil data from two sources:

1. **Tabular attributes** — queried directly from the USDA Soil Data Access (SDA)
   REST API. Returns HSG, drainage class, Ksat, and AWC for all 11,208 Iowa
   map units without downloading any large files.

2. **Spatial map unit polygons** — downloaded as per-county SSURGO ZIPs from
   Web Soil Survey (one ZIP per county, 99 counties). Only the map unit
   shapefile is extracted from each ZIP; the tabular files inside are discarded
   since the SDA query covers them. The 99 county shapefiles are merged into a
   single Iowa file.

**Outputs**
- `data/tabular/soil/raw/ssurgo-iowa-attributes.csv` — map unit key + 4 soil attributes
- `data/spatial/ssurgo/iowa-mapunit-polygons.shp` — merged Iowa map unit polygons

**Resuming**  
Already-downloaded county ZIPs are detected by checking for the county's
shapefile in a scratch folder, so the spatial download can be interrupted
and restarted.

In [1]:
import io
import time
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path

SDA_URL      = 'https://sdmdataaccess.sc.egov.usda.gov/Tabular/SDMTabularService/post.rest'
WSS_URL      = 'https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_{sym}_[{date}].zip'
EXPORT_DATE  = '2025-09-05'   # current export date for all Iowa survey areas
SCRATCH_DIR  = Path('/tmp/ssurgo_iowa')
OUT_CSV      = Path('../../data/tabular/soil/raw/ssurgo-iowa-attributes.csv')
OUT_SHP      = Path('../../data/spatial/ssurgo/iowa-mapunit-polygons.shp')
PAUSE_SEC    = 0.5

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Tabular attributes via SDA API

In [2]:
# Dominant component per map unit: HSG, drainage class, Ksat, AWC
# Horizon values are depth-weighted to represent the full profile
sql = """
SELECT
    mu.mukey,
    mu.muname,
    mu.musym,
    l.areasymbol,
    c.compname,
    c.comppct_r,
    c.hydgrp,
    c.drainagecl,
    AVG(ch.ksat_r)  AS ksat_r_mean,
    AVG(ch.awc_r)   AS awc_r_mean
FROM legend l
JOIN mapunit mu  ON l.lkey  = mu.lkey
JOIN component c ON mu.mukey = c.mukey
JOIN chorizon ch ON c.cokey  = ch.cokey
WHERE l.areasymbol LIKE 'IA%'
  AND c.majcompflag = 'Yes'
GROUP BY mu.mukey, mu.muname, mu.musym, l.areasymbol,
         c.compname, c.comppct_r, c.hydgrp, c.drainagecl
ORDER BY mu.mukey
"""

print('Querying SDA for Iowa soil attributes...')
resp = requests.post(SDA_URL, data={'query': sql, 'format': 'JSON+COLUMNNAME'}, timeout=120)
resp.raise_for_status()
data = resp.json()['Table']
cols, rows = data[0], data[1:]
df = pd.DataFrame(rows, columns=cols)
df[['comppct_r', 'ksat_r_mean', 'awc_r_mean']] = df[['comppct_r', 'ksat_r_mean', 'awc_r_mean']].apply(pd.to_numeric, errors='coerce')

print(f'  Raw rows: {len(df):,}  (one per map unit × component × horizon group)')

# Keep only the dominant component per map unit (highest comppct_r)
df = df.sort_values('comppct_r', ascending=False).drop_duplicates('mukey')
print(f'  After dominant-component filter: {len(df):,} map units')

df.to_csv(OUT_CSV, index=False)
print(f'  Saved → {OUT_CSV}')
df.head()

Querying SDA for Iowa soil attributes...


  Raw rows: 11,793  (one per map unit × component × horizon group)
  After dominant-component filter: 10,572 map units
  Saved → ../../data/tabular/soil/raw/ssurgo-iowa-attributes.csv


,mukey,muname,musym,areasymbol,compname,comppct_r,hydgrp,drainagecl,ksat_r_mean,awc_r_mean
11792,3471453,"Anthroportic Udorthents, mine spoil, 0 to 60 p...",5012,IA181,Anthroportic Udorthents,100,C,Well drained,1.85,0.10
3853,406966,"Renova loam, 2 to 5 percent slopes",491B,IA089,Renova,100,B,Well drained,9.00,0.20
6772,410478,"Salix silty clay loam, 0 to 2 percent slopes",36,IA155,Salix,100,C,Moderately well drained,5.40,0.21
6771,410477,"Steinauer clay loam, 14 to 18 percent slopes, ...",33E2,IA155,Steinauer,100,C,Well drained,3.00,0.17
6770,410476,"Steinauer clay loam, 9 to 14 percent slopes, m...",33D2,IA155,Steinauer,100,C,Well drained,3.00,0.17


## 2. Spatial map unit polygons (county-by-county WSS download)

In [3]:
# Iowa survey areas: IA001, IA003, ..., IA197  (99 counties, odd FIPS)
ia_symbols = [f'IA{str(i).zfill(3)}' for i in range(1, 198, 2)]
print(f'Survey areas to download: {len(ia_symbols)}')

county_gdfs = []
failed = []

for i, sym in enumerate(ia_symbols):
    sym_lower = sym.lower()
    shp_name  = f'soilmu_a_{sym_lower}.shp'
    scratch_shp = SCRATCH_DIR / sym / shp_name

    # Resume: skip if already extracted
    if scratch_shp.exists():
        gdf = gpd.read_file(scratch_shp)
        gdf['areasymbol'] = sym
        county_gdfs.append(gdf)
        continue

    url = WSS_URL.format(sym=sym, date=EXPORT_DATE)
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f'  WARN {sym}: {e}')
        failed.append(sym)
        continue

    county_dir = SCRATCH_DIR / sym
    county_dir.mkdir(exist_ok=True)

    # Extract only spatial/soilmu_a_<sym>.* files (polygon layer)
    # ZIP structure: {SYM}/spatial/soilmu_a_{sym_lower}.*
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        spatial_files = [
            n for n in z.namelist()
            if f'spatial/soilmu_a_{sym_lower}' in n.lower()
        ]
        for name in spatial_files:
            target = county_dir / Path(name).name
            target.write_bytes(z.read(name))

    if not scratch_shp.exists():
        print(f'  WARN {sym}: {shp_name} not found in ZIP')
        failed.append(sym)
        continue

    gdf = gpd.read_file(scratch_shp)
    gdf['areasymbol'] = sym
    county_gdfs.append(gdf)
    time.sleep(PAUSE_SEC)

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/{len(ia_symbols)}] {sym} done — {len(county_gdfs)} loaded')

print(f'\nLoaded {len(county_gdfs)} counties. Failed: {failed}')

Survey areas to download: 99


  WARN IA011: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA011_%5B2025-09-05%5D.zip


  WARN IA013: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA013_%5B2025-09-05%5D.zip


  WARN IA015: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA015_%5B2025-09-05%5D.zip


  WARN IA017: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA017_%5B2025-09-05%5D.zip


...

  WARN IA189: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA189_%5B2025-09-05%5D.zip


  WARN IA191: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA191_%5B2025-09-05%5D.zip


  WARN IA193: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA193_%5B2025-09-05%5D.zip


  WARN IA195: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA195_%5B2025-09-05%5D.zip


  WARN IA197: 400 Client Error: Bad Request for url: https://websoilsurvey.sc.egov.usda.gov/DSD/Download/Cache/SSA/wss_SSA_IA197_%5B2025-09-05%5D.zip

Loaded 5 counties. Failed: ['IA011', 'IA013', 'IA015', 'IA017', 'IA019', 'IA021', 'IA023', 'IA025', 'IA027', 'IA029', 'IA031', 'IA033', 'IA035', 'IA037', 'IA039', 'IA041', 'IA043', 'IA045', 'IA047', 'IA049', 'IA051', 'IA053', 'IA055', 'IA057', 'IA059', 'IA061', 'IA063', 'IA065', 'IA067', 'IA069', 'IA071', 'IA073', 'IA075', 'IA077', 'IA079', 'IA081', 'IA083', 'IA085', 'IA087', 'IA089', 'IA091', 'IA093', 'IA095', 'IA097', 'IA099', 'IA101', 'IA103', 'IA105', 'IA107', 'IA109', 'IA111', 'IA113', 'IA115', 'IA117', 'IA119', 'IA121', 'IA123', 'IA125', 'IA127', 'IA129', 'IA131', 'IA133', 'IA135', 'IA137', 'IA139', 'IA141', 'IA143', 'IA145', 'IA147', 'IA149', 'IA151', 'IA153', 'IA155', 'IA157', 'IA159', 'IA161', 'IA163', 'IA165', 'IA167', 'IA169', 'IA171', 'IA173', 'IA175', 'IA177', 'IA179', 'IA181', 'IA183', 'IA185', 'IA187', 'IA189', 'IA191', 'I

## 3. Merge county shapefiles and save

In [4]:
iowa = pd.concat(county_gdfs, ignore_index=True)
print(f'Total map unit polygons: {len(iowa):,}')
print(f'CRS: {iowa.crs}')

OUT_SHP.parent.mkdir(parents=True, exist_ok=True)
iowa.to_file(OUT_SHP)
print(f'Saved → {OUT_SHP}')

# Clean up scratch
shutil.rmtree(SCRATCH_DIR)
print('Scratch directory removed.')

Total map unit polygons: 129,748
CRS: EPSG:4326


Saved → ../../data/spatial/ssurgo/iowa-mapunit-polygons.shp
Scratch directory removed.


## 4. Quick summary

In [5]:
attrs = pd.read_csv(OUT_CSV)
print('=== Tabular attributes ===')
print(f'Map units: {len(attrs):,}')
print(f'HSG distribution:\n{attrs["hydgrp"].value_counts()}')
print(f'Drainage class distribution:\n{attrs["drainagecl"].value_counts().head()}')
print(f'Ksat (mm/hr) — mean: {attrs["ksat_r_mean"].mean():.2f}, median: {attrs["ksat_r_mean"].median():.2f}')
print(f'AWC (cm/cm)  — mean: {attrs["awc_r_mean"].mean():.3f}, median: {attrs["awc_r_mean"].median():.3f}')
print()
spatial = gpd.read_file(OUT_SHP)
print('=== Spatial polygons ===')
print(f'Polygons: {len(spatial):,}')
print(f'CRS: {spatial.crs}')
spatial.head()

=== Tabular attributes ===
Map units: 10,572
HSG distribution:
hydgrp
C      3782
C/D    2013
B      1753
D      1497
A       961
B/D     486
A/D      63
Name: count, dtype: int64
Drainage class distribution:
drainagecl
Well drained               4229
Moderately well drained    1892
Somewhat poorly drained    1720
Poorly drained             1616
Excessively drained         484
Name: count, dtype: int64
Ksat (mm/hr) — mean: 17.28, median: 6.00
AWC (cm/cm)  — mean: 0.172, median: 0.180



=== Spatial polygons ===
Polygons: 129,748
CRS: EPSG:4326


,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,areasymb_1,geometry
0,IA001,9,370C2,402178,IA001,"POLYGON ((-94.42495 41.26683, -94.4249 41.2668..."
1,IA001,9,Y24D2,402163,IA001,"POLYGON ((-94.60511 41.26692, -94.60493 41.267..."
2,IA001,9,Y24D2,402163,IA001,"POLYGON ((-94.6842 41.26772, -94.68399 41.2678..."
3,IA001,9,11B,402143,IA001,"POLYGON ((-94.53318 41.25107, -94.533 41.25103..."
4,IA001,9,428B,402182,IA001,"POLYGON ((-94.53727 41.2493, -94.53739 41.2498..."
